# Campus Placement Prediction — Exploratory Data Analysis

This notebook performs comprehensive EDA on the synthetic campus placement dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_generator import generate_placement_data
from src.feature_engineering import add_engineered_features, encode_performance_tier

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

## 1. Load Data

In [ ]:
data_path = os.path.join('..', 'data', 'placement_data.csv')
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    df = generate_placement_data(1000)
    os.makedirs(os.path.dirname(data_path), exist_ok=True)
    df.to_csv(data_path, index=False)

print(f'Shape: {df.shape}')
df.head()

## 2. Basic Statistics

In [ ]:
df.describe(include='all')

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nPlacement rate: {df["placement_status"].mean():.1%}')
print(f'Avg salary (placed): {df["salary"].mean():.2f} LPA')

## 3. Placement Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall placement
counts = df['placement_status'].value_counts()
axes[0].pie(counts, labels=['Placed', 'Not Placed'], autopct='%1.1f%%',
             colors=['#27ae60', '#e74c3c'], startangle=90)
axes[0].set_title('Placement Distribution')

# By gender
gender_grp = df.groupby('gender')['placement_status'].mean() * 100
gender_grp.plot(kind='bar', ax=axes[1], color=['#3498db', '#e91e8c'], edgecolor='black')
axes[1].set_title('Placement Rate by Gender (%)')
axes[1].set_ylabel('Placement Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 4. Academic Performance Analysis

In [ ]:
acad_cols = ['ssc_percentage', 'hsc_percentage', 'degree_percentage',
              'etest_percentage', 'mba_percentage']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(acad_cols):
    df[df['placement_status']==1][col].hist(ax=axes[i], bins=25, alpha=0.7,
                                             color='#27ae60', label='Placed')
    df[df['placement_status']==0][col].hist(ax=axes[i], bins=25, alpha=0.7,
                                             color='#e74c3c', label='Not Placed')
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].legend()

axes[5].set_visible(False)
plt.suptitle('Academic Score Distributions by Placement Status', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Categorical Variables Analysis

In [ ]:
cat_cols = ['hsc_subject', 'degree_type', 'mba_specialization', 'work_experience']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    grp = df.groupby(col)['placement_status'].mean() * 100
    grp.sort_values(ascending=True).plot(
        kind='barh', ax=axes[i],
        color=sns.color_palette('Blues_d', len(grp)), edgecolor='black'
    )
    axes[i].set_title(f'Placement Rate by {col.replace("_", " ").title()} (%)')
    axes[i].set_xlabel('Placement Rate (%)')

plt.tight_layout()
plt.show()

## 6. Salary Analysis

In [ ]:
placed_df = df[df['placement_status'] == 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Salary distribution
placed_df['salary'].hist(bins=30, ax=axes[0], color='#2a5298', edgecolor='white')
axes[0].set_title('Salary Distribution (LPA)')
axes[0].set_xlabel('Salary (LPA)')

# By specialization
placed_df.groupby('mba_specialization')['salary'].mean().plot(
    kind='bar', ax=axes[1], color=['#27ae60', '#e67e22'], edgecolor='black'
)
axes[1].set_title('Avg Salary by MBA Specialization')
axes[1].set_ylabel('Avg Salary (LPA)')
axes[1].tick_params(axis='x', rotation=15)

# By work experience
placed_df.groupby('work_experience')['salary'].mean().plot(
    kind='bar', ax=axes[2], color=['#3498db', '#e74c3c'], edgecolor='black'
)
axes[2].set_title('Avg Salary by Work Experience')
axes[2].set_ylabel('Avg Salary (LPA)')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
num_cols = ['ssc_percentage', 'hsc_percentage', 'degree_percentage',
             'etest_percentage', 'mba_percentage', 'placement_status', 'salary']
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
             square=True, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Engineered Features Analysis

In [ ]:
df_feat = add_engineered_features(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Academic composite by placement
for status, label, color in [(1, 'Placed', '#27ae60'), (0, 'Not Placed', '#e74c3c')]:
    df_feat[df_feat['placement_status']==status]['academic_composite'].hist(
        bins=25, alpha=0.7, ax=axes[0], color=color, label=label
    )
axes[0].set_title('Academic Composite Score')
axes[0].legend()

# Placement rate by performance tier
tier_grp = df_feat.groupby('performance_tier')['placement_status'].mean() * 100
tier_order = ['Low', 'Mid', 'High']
tier_grp = tier_grp.reindex([t for t in tier_order if t in tier_grp.index])
tier_grp.plot(kind='bar', ax=axes[1],
               color=['#e74c3c', '#f39c12', '#27ae60'], edgecolor='black')
axes[1].set_title('Placement Rate by Performance Tier (%)')
axes[1].set_ylabel('Placement Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('\nNew features preview:')
df_feat[['academic_composite', 'academic_consistency', 'degree_to_mba_growth',
          'performance_tier', 'exp_fin_interaction', 'top_performer']].describe()

## 9. Key Insights Summary

In [ ]:
print('=' * 55)
print('  KEY INSIGHTS FROM EDA')
print('=' * 55)
print(f'1. Overall placement rate: {df["placement_status"].mean():.1%}')
print(f'2. Gender: Male placement rate: '
      f'{df[df["gender"]=="Male"]["placement_status"].mean():.1%}, '
      f'Female: {df[df["gender"]=="Female"]["placement_status"].mean():.1%}')
print(f'3. Work experience boosts placement: '
      f'{df[df["work_experience"]=="Yes"]["placement_status"].mean():.1%} vs '
      f'{df[df["work_experience"]=="No"]["placement_status"].mean():.1%}')
print(f'4. Mkt&Fin avg salary: '
      f'{placed_df[placed_df["mba_specialization"]=="Mkt&Fin"]["salary"].mean():.2f} LPA')
print(f'   Mkt&HR avg salary: '
      f'{placed_df[placed_df["mba_specialization"]=="Mkt&HR"]["salary"].mean():.2f} LPA')
print(f'5. Top academic correlation with placement: '
      f'{df[["ssc_percentage","hsc_percentage","degree_percentage","mba_percentage","etest_percentage"]].corrwith(df["placement_status"]).idxmax()}')
print('=' * 55)